# Latent-class logit

## Model

Choice probabilities are a finite mixture of class-specific logits:

$$
P_{nj}=\sum_{c=1}^{2}\pi_{nc}P_{nj\mid c},\qquad
P_{nj\mid c}=
\frac{\exp(V_{njc})}{\sum_{k\in\mathcal C_n}\exp(V_{nkc})}.
$$

Class 1 is the membership reference. Class-2 membership depends on income and
traveler status:

$$
\pi_{n2}=
\frac{\exp(\delta_0+\delta_1\,\mathrm{income}_n
+\delta_2\,\mathrm{frequent}_n)}
{1+\exp(\delta_0+\delta_1\,\mathrm{income}_n
+\delta_2\,\mathrm{frequent}_n)}.
$$

Both classes have their own alternative constants, time coefficient, and cost
coefficient. The example therefore shows observed heterogeneity in class
membership and unobserved heterogeneity in class-specific tastes.

This notebook is self-contained and was executed in the repository's Office
validation environment. Install the released package with
`pip install torchdcm`, then select CPU or CUDA through the `device` argument.


In [1]:
import numpy as np
import torch

from IPython.display import HTML, display

from torchdcm import Beta, ChoiceDataset, LatentClassLogit, UtilitySpec
from torchdcm.datasets import make_swissmetro_like
torch.set_default_dtype(torch.float64)
torch.manual_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TorchDCM example running on {device}")


TorchDCM example running on cuda


In [2]:
rng = np.random.default_rng(20)
frame = make_swissmetro_like(n_obs=700, seed=20)
alternatives = np.array(["TRAIN", "SM", "CAR"])
time = frame[["time_train", "time_sm", "time_car"]].to_numpy()
cost = frame[["cost_train", "cost_sm", "cost_car"]].to_numpy()
available = frame[["avail_train", "avail_sm", "avail_car"]].to_numpy(bool)
frame["income_std"] = (
    (frame["income"] - frame["income"].mean()) / frame["income"].std()
)
frame["frequent_traveler"] = (frame["person_id"] % 2 == 0).astype(float)

membership_index = (
    -0.30
    + 0.80 * frame["income_std"].to_numpy()
    + 0.60 * frame["frequent_traveler"].to_numpy()
)
p_class_2 = 1.0 / (1.0 + np.exp(-membership_index))
class_2 = rng.uniform(size=len(frame)) < p_class_2

asc = np.where(
    class_2[:, None],
    np.array([[-0.20, 0.0, 0.30]]),
    np.array([[0.40, 0.0, 0.10]]),
)
b_time = np.where(class_2, -0.045, -0.018)
b_cost = np.where(class_2, -0.030, -0.090)
utility = asc + b_time[:, None] * time + b_cost[:, None] * cost
masked = np.where(available, utility, -np.inf)
probabilities = np.exp(masked - np.max(masked, axis=1, keepdims=True))
probabilities /= probabilities.sum(axis=1, keepdims=True)
frame["choice"] = [
    rng.choice(alternatives, p=row) for row in probabilities
]

data = ChoiceDataset.from_wide(
    frame,
    alternatives=alternatives.tolist(),
    choice="choice",
    variables={
        "time": {"TRAIN": "time_train", "SM": "time_sm", "CAR": "time_car"},
        "cost": {"TRAIN": "cost_train", "SM": "cost_sm", "CAR": "cost_car"},
        "income_std": {
            "TRAIN": "income_std",
            "SM": "income_std",
            "CAR": "income_std",
        },
        "frequent_traveler": {
            "TRAIN": "frequent_traveler",
            "SM": "frequent_traveler",
            "CAR": "frequent_traveler",
        },
    },
    availability={
        "TRAIN": "avail_train",
        "SM": "avail_sm",
        "CAR": "avail_car",
    },
    individual_id="person_id",
)


def class_spec(label, *, time_init, cost_init):
    spec = UtilitySpec()
    spec.utility(
        "TRAIN",
        Beta(f"ASC_TRAIN_{label}", init=0.0)
        + Beta(f"B_TIME_{label}", init=time_init) * "time"
        + Beta(f"B_COST_{label}", init=cost_init) * "cost",
    )
    spec.utility(
        "SM",
        Beta(f"B_TIME_{label}", init=time_init) * "time"
        + Beta(f"B_COST_{label}", init=cost_init) * "cost",
    )
    spec.utility(
        "CAR",
        Beta(f"ASC_CAR_{label}", init=0.0)
        + Beta(f"B_TIME_{label}", init=time_init) * "time"
        + Beta(f"B_COST_{label}", init=cost_init) * "cost",
    )
    return spec


In [3]:
class_specs = [
    class_spec("CLASS_1", time_init=-0.020, cost_init=-0.070),
    class_spec("CLASS_2", time_init=-0.040, cost_init=-0.040),
]
membership_class_2 = (
    Beta("CLASS_2_INTERCEPT", init=-0.10)
    + Beta("CLASS_2_INCOME", init=0.20) * "income_std"
    + Beta("CLASS_2_FREQUENT", init=0.20) * "frequent_traveler"
)
model = LatentClassLogit(
    class_specs,
    class_membership=[membership_class_2],
    device=device,
    max_iter=100,
)
result = model.fit(data)
display(HTML(result.report().to_html()))


Model family,LatentClassLogit
Estimation objective,Maximum log likelihood
TorchDCM version,0.1.0
PyTorch version,2.12.1+cu130
Python version,3.12.13
Operating system,Linux-6.17.0-35-generic-x86_64-with-glibc2.39
Device,cuda
Tensor dtype,float64
Optimizer,torch.optim.LBFGS
Maximum iterations,100
Line search,strong_wolfe
